# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

From *The State of AI-Driven SEO* (FlyRank, March 2026, in `docs/`). The paper is careful in many places (it calls its ML pages exploratory and flags its own tiny buckets), so these are questions about how to make two findings stronger, not a takedown.

**Finding #1, "The Anatomy of Growing Content" (CONFIRMED in the paper).** Pages with rising impressions are on average longer (3.2K vs 2.3K words) and younger (184 vs 230 days) than pages with falling impressions.
- *Where the label comes from:* "rising" and "falling" come from the 30-day vs previous-30-day impression change. That is an observed outcome, but it sits inside the same 90-day snapshot as the word counts and ages, so it describes pages that were already moving, it does not forecast.
- *Does the validation carry the claim?* There is no validation step; it is a comparison of two group averages, pooled across 57 brands. If a few big brands both write longer pages and happened to grow that quarter, the gap could be a brand effect, not a page effect.
- *My question:* does the word-count gap hold **within** each brand (compare inside a client, then average), and does word count measured *before* a window predict growth *in* that window? I also noticed that `word_count` is a current snapshot, so for pages that were refreshed (and lengthened) during the window, "longer pages grow" could partly be "pages that were just expanded grow".

**Finding #4, "The Freshness Multiplier" (CONFIRMED in the paper).** "365+ day content that was refreshed within 30 days shows 3.2× health boost (10.7 → 34.5) and 57× more impressions (71 → 4,039)", turned into the advice "Refreshing mature pages produces 3.2× health and 57× impressions".
- *Where the label comes from:* the health score is built *from* impressions, position, CTR and scroll, so "health" and "impressions" are not two separate outcomes. Part of the 3.2× is the 57× counted again.
- *Does the validation carry the claim?* The refreshed group was **chosen**: editors usually refresh pages that matter, still rank, or have demand. Comparing them to untouched old pages mixes the effect of refreshing with the effect of *which pages got picked*. Without a before/after on the same pages, or a matched comparison, the numbers are an association, and "produces" is a causal word the design cannot support.
- *My question:* what were these same pages' impressions in the 30-60 days *before* the refresh? If they were already high, the refresh followed strength rather than caused it. A matched set of similar old pages that were not refreshed would turn this into a much stronger finding.

Both questions apply to my own work too, which is what sections 2-4 check.

In [1]:
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# numpy 2 + Apple Accelerate prints spurious "overflow in matmul" warnings inside sklearn; results are unaffected
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sohamrathi17/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("work/scripts") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, "work/scripts")

import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
import lane_data as L

# Test month (same as ML-08): decide 31 March, label April
mar = L.add_drop_feature(L.build_frame("2026-03", "2026-04"))
# An EARLIER month for the time-aware check: decide 28 February, label March
feb = L.add_drop_feature(L.build_frame("2026-02", "2026-03"))
for name, fr in [("Feb -> Mar", feb), ("Mar -> Apr", mar)]:
    print(f"{name}: {len(fr):,} pages, {fr['client_hash_id'].nunique()} clients, declining {fr['is_declining'].mean():.1%}")

Feb -> Mar: 43,596 pages, 23 clients, declining 21.1%
Mar -> Apr: 50,299 pages, 32 clients, declining 47.4%


## 2. My model under an honest split (before/after)

My carried-forward score from ML-08 is **logistic probability × March drop per day**. Here it is under three designs, from least to most honest:

1. **Before (random row split):** 5 folds over pages, ignoring clients. Pages from the same site sit on both sides.
2. **After, grouped (ML-08 design):** 5 folds grouped by client. Every page is scored by a model that never saw its site.
3. **After, time-aware:** fit on the *February → March* frame, then score the *March → April* frame. The model never saw the test month at all, and I did not tune anything on it. This is the closest to real use: train on last month, rank this month.

Same metrics as always: macro precision@20 and lost impressions captured in the top 20, over the 22 clients with at least 20 pages, plus ROC-AUC for the probability. The ML-07 rule is the reference row.

In [2]:
from sklearn.model_selection import KFold

y, drop = mar["is_declining"], L.drop_per_day(mar)
X = mar[L.MODEL_FEATURES]

# 1) random row split
p_random = np.zeros(len(mar))
for tr, te in KFold(5, shuffle=True, random_state=L.SEED).split(X):
    p_random[te] = L.make_logistic().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]

# 2) grouped by client
p_grouped = L.out_of_fold(mar, L.make_logistic, "is_declining")

# 3) time-aware: train on Feb -> Mar, test on Mar -> Apr
time_model = L.make_logistic().fit(feb[L.MODEL_FEATURES], feb["is_declining"])
p_time = time_model.predict_proba(X)[:, 1]

rows = [{"design": "ML-07 rule (no training)", **L.summarize(mar, L.baseline_score(mar)), "ROC-AUC": np.nan}]
for name, p in [("before: random row split", p_random), ("after: grouped by client", p_grouped),
                ("after: time-aware (Feb->Mar train)", p_time)]:
    rows.append({"design": name, **L.summarize(mar, p * (1 + drop)), "ROC-AUC": roc_auc_score(y, p)})
splits = pd.DataFrame(rows).set_index("design")[["precision@20", "loss_captured@20", "ROC-AUC", "base_rate"]]
splits.round(3)

,precision@20,loss_captured@20,ROC-AUC,base_rate
design,,,,
ML-07 rule (no training),0.711,0.241,NaN,0.489
before: random row split,0.716,0.242,0.723,0.489
after: grouped by client,0.718,0.242,0.633,0.489
after: time-aware (Feb->Mar train),0.720,0.235,0.658,0.489


In [3]:
# Time-aware AND unseen clients: clients that appear in March but were not in the February training frame
new_clients = ~mar["client_hash_id"].isin(feb["client_hash_id"])
sub = mar[new_clients]
print(f"Clients in the test month that the time-aware model never saw at all: {sub['client_hash_id'].nunique()} "
      f"({len(sub):,} pages)")
m_new = L.summarize(sub, p_time[new_clients.values] * (1 + drop[new_clients]))
m_rule = L.summarize(sub, L.baseline_score(mar)[new_clients])
print(f"  evaluated (>= 20 pages): {m_new['clients']} clients")
print(f"  time-aware model: precision@20 {m_new['precision@20']:.3f}, loss captured {m_new['loss_captured@20']:.3f}")
print(f"  ML-07 rule:       precision@20 {m_rule['precision@20']:.3f}, loss captured {m_rule['loss_captured@20']:.3f}")

# Calibration: are the probabilities themselves trustworthy across months?
print(f"\nAverage predicted probability (time-aware model) on the April label: {p_time.mean():.3f}")
print(f"Actual April declining rate:                                      {y.mean():.3f}")
print(f"February -> March declining rate the model was trained on:        {feb['is_declining'].mean():.3f}")

Clients in the test month that the time-aware model never saw at all: 9 (6,630 pages)
  evaluated (>= 20 pages): 8 clients
  time-aware model: precision@20 0.644, loss captured 0.233
  ML-07 rule:       precision@20 0.612, loss captured 0.245

Average predicted probability (time-aware model) on the April label: 0.245
Actual April declining rate:                                      0.474
February -> March declining rate the model was trained on:        0.211


**What the before/after shows:**

- **The random split flatters the probability.** ROC-AUC drops by roughly 9 points from the random split to the grouped split. That gap is what the model gains by recognising a site it has already seen, and it would not exist for a new client. This is the same lesson as ML-01, now on warehouse data.
- **The top-20 queue barely moves across all three designs.** Most of the ranking comes from the size of the March drop, which needs no training, so it cannot memorise anything. That is reassuring for the queue and a reminder that the model's own contribution is modest.
- **It survives a month it was never tuned on.** Trained on February→March and tested on March→April, the score keeps its precision@20 at about the ML-08 level and stays above the rule, with similar loss coverage. The variant choice I made in ML-08 after seeing the table was not just a lucky pick for one month.
- **But the probabilities do not transfer.** The decline rate jumped from about 21% (Feb→Mar) to about 47% (Mar→Apr), and the time-aware model's average prediction stays near the old rate. The scores are fine **for ordering pages within a month** and wrong **as probabilities**. The queue must never say "72% chance this page declines", only "ranked 3rd for review".
- **On clients that are new to the model, the result is from only a few clients** (see the printout), so I treat it as directional.

## 3. Leakage audit

The checklist from the leakage skill, run on my final feature set. Each item has a check below.

| Check | How | Result |
|---|---|---|
| Timeline: all features before the label window | features built only from the feature month's partition and creation dates; asserted below | see code |
| No label-derived or sibling columns | the label sources (`imp_next`, `is_declining`, `impressions_lost`) are not in the feature list; asserted | see code |
| Deliberate leak raises the score toward 1.0 | add next-month impressions per day, grouped CV, then remove | see code |
| No product flags or existing-system scores | the warehouse release has no health score or flags, and I use none | by design |
| Snapshot fields | `content_updated_date`, `last_optimized_date` excluded (ML-04: 78% postdate the decision). Keyword metadata is also a snapshot, so I train *without* it and compare | see code |
| Population selection uses outcome-window info | the slice requires 30 days of April GSC data, so it knows the page still existed in April. Quantified below and disclosed | see code |
| Split grouped by client and time-aware | section 2 | done |
| Base rate next to every metric | every table here and in ML-08 | done |
| Metrics out-of-fold, never in-sample | grouped OOF and a separate month | done |

In [4]:
# Timeline + label-sibling checks
assert not set(L.NEVER_FEATURES) & set(L.MODEL_FEATURES), "a never-feature is in the feature list"
for fr, decision in [(mar, "2026-03-31"), (feb, "2026-02-28")]:
    assert (fr["decision_date"] == pd.Timestamp(decision)).all()
    assert (fr["age_days"] >= 0).all(), "a page created after the decision moment"
print("Features used:", L.MODEL_FEATURES)
print("None of the label columns or snapshot dates is among them: OK")

# Deliberate leak: add next-month impressions per day, score it with grouped CV, then remove it
from sklearn.model_selection import GroupKFold
leak_cols = L.MODEL_FEATURES + ["imp_next_per_day"]
mar["imp_next_per_day"] = mar["imp_next"] / 30
p_leak = np.zeros(len(mar))
for tr, te in GroupKFold(5).split(mar, groups=mar["client_hash_id"]):
    p_leak[te] = (L.make_boosted_classifier().fit(mar.iloc[tr][leak_cols], y.iloc[tr])
                  .predict_proba(mar.iloc[te][leak_cols])[:, 1])
mar = mar.drop(columns=["imp_next_per_day"])
print(f"\nWITH deliberate leak:  ROC-AUC {roc_auc_score(y, p_leak):.3f}  <- the harness catches it")
print(f"WITHOUT (grouped CV):  ROC-AUC {roc_auc_score(y, p_grouped):.3f}  <- the honest number")

Features used: ['log_imp', 'avg_position', 'ctr', 'age_days', 'momentum', 'log_clicks', 'days_with_imp', 'imp_cv', 'position_change', 'top_day_share', 'log_search_volume', 'has_search_volume', 'competition', 'keyword_token_count', 'log_drop', 'content_type', 'main_intent']
None of the label columns or snapshot dates is among them: OK



WITH deliberate leak:  ROC-AUC 0.999  <- the harness catches it
WITHOUT (grouped CV):  ROC-AUC 0.633  <- the honest number


In [5]:
# Snapshot suspects: keyword metadata from dim_content. Train WITHOUT them and compare.
SUSPECTS = ["log_search_volume", "has_search_volume", "competition", "keyword_token_count", "content_type", "main_intent"]
kept_num = [c for c in L.MODEL_NUMERIC if c not in SUSPECTS]

from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
p_wo = np.zeros(len(mar))
for tr, te in GroupKFold(5).split(mar, groups=mar["client_hash_id"]):
    model = make_pipeline(make_column_transformer((make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), kept_num)),
                          LogisticRegression(max_iter=2000))
    p_wo[te] = model.fit(mar.iloc[tr][kept_num], y.iloc[tr]).predict_proba(mar.iloc[te][kept_num])[:, 1]
w, wo = L.summarize(mar, p_grouped * (1 + drop)), L.summarize(mar, p_wo * (1 + drop))
print(f"with keyword metadata:    ROC-AUC {roc_auc_score(y, p_grouped):.3f} | precision@20 {w['precision@20']:.3f} | loss captured {w['loss_captured@20']:.3f}")
print(f"without keyword metadata: ROC-AUC {roc_auc_score(y, p_wo):.3f} | precision@20 {wo['precision@20']:.3f} | loss captured {wo['loss_captured@20']:.3f}")

with keyword metadata:    ROC-AUC 0.633 | precision@20 0.718 | loss captured 0.242
without keyword metadata: ROC-AUC 0.624 | precision@20 0.709 | loss captured 0.240


In [6]:
# Population selection: how many otherwise-eligible March pages were dropped because April coverage was incomplete?
import duckdb
tok = L.get_token()
MAR_P, APR_P = L.month_path("2026-03", tok), L.month_path("2026-04", tok)
eligible = duckdb.sql(f"""
    WITH m AS (SELECT client_hash_id, content_hash_id FROM read_parquet('{MAR_P}')
               WHERE gsc_data_available IS TRUE GROUP BY ALL
               HAVING COUNT(*) = 31 AND SUM(gsc_impressions) >= {L.VOLUME_FLOOR}),
         a AS (SELECT client_hash_id, content_hash_id, COUNT(*) AS apr_days FROM read_parquet('{APR_P}')
               WHERE gsc_data_available IS TRUE GROUP BY ALL)
    SELECT COUNT(*) AS march_eligible,
           COUNT(*) FILTER (WHERE a.apr_days = 30) AS kept,
           COUNT(*) FILTER (WHERE a.apr_days IS NULL) AS no_april_gsc_rows,
           COUNT(*) FILTER (WHERE a.apr_days BETWEEN 1 AND 29) AS partial_april
    FROM m LEFT JOIN a USING (client_hash_id, content_hash_id)
""").df()
eligible["dropped_share"] = 1 - eligible["kept"] / eligible["march_eligible"]
eligible.round(3)

,march_eligible,kept,no_april_gsc_rows,partial_april,dropped_share
0,61711,50299,12,11400,0.185


**Audit result:**

- **Timeline and label checks pass.** No April column, label column or snapshot date is a feature, and every page was created before its decision moment.
- **The harness catches a leak.** Adding next-month impressions sends ROC-AUC to about 1.0. I removed it, and the honest grouped number stays where ML-08 had it.
- **Keyword metadata is not carrying the result.** Dropping all six snapshot-based keyword fields changes the numbers very little (printout above), so even if that metadata were partly updated after March, it is not secretly driving the score.
- **One real, disclosed selection effect.** The slice keeps only pages with full April GSC coverage, which uses information from the outcome window. It drops about 18.5% of March-eligible pages, almost all because they have only *some* April days of GSC data. Many of those are probably tracking gaps, but some may be pages that faded or were removed. So my decline rate could be an *under*-count, and the queue was never tested on pages that lost coverage. This is a choice, not a leak in the features, and it goes into the paper's limitations.

## 4. Claim rewrite

**My boldest sentence so far** (my own draft summary after ML-08):

> "The model beats the hand-written rule and predicts which pages will decline next month."

**What is wrong with it:**
- "beats" hides a trade-off: the model wins on precision@20 and matches the rule on lost impressions captured, by a few points, for clients with a few hundred pages or more.
- "predicts which pages will decline" sounds like a reliable forecast for any page. Its probabilities do not transfer between months (section 2), and ROC-AUC is about 0.63-0.66.
- It leaves out the population: one decision month and one test month, 22 evaluated clients, pages with full GSC coverage.

**Rewrite (safe language):**

> "In this data (pages with full Search Console coverage, ranked within each of 22 clients), a score combining a logistic model's decline probability with the size of each page's March impression drop put about 72% true April decliners in each client's top 20, against a 49% base rate and 71% for a transparent hand-written rule, and kept that level when trained on an earlier month. It is a decision-support ranking for choosing which pages to review first, not a forecast of individual pages, and it says nothing about what a refresh would cause."

In [7]:
results = {
    "assignment": "ML-09 validation audit",
    "test_slice": "decision 2026-03-31, label April 2026",
    "time_train_slice": "decision 2026-02-28, label March 2026",
    "score": "logistic probability x (1 + March drop per day)",
    "seed": L.SEED,
    "designs": {k: {c: (None if pd.isna(v) else round(float(v), 4)) for c, v in r.items()}
                for k, r in splits.to_dict("index").items()},
    "new_clients_time_aware": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m_new.items()},
    "calibration": {"mean_pred_time_model": round(float(p_time.mean()), 4), "actual_rate": round(float(y.mean()), 4),
                    "train_month_rate": round(float(feb["is_declining"].mean()), 4)},
    "leak_test_auc": round(float(roc_auc_score(y, p_leak)), 4),
    "population_dropped_share": round(float(eligible["dropped_share"].iloc[0]), 4),
}
with open("work/outputs/validation_metrics.json", "w") as fh:
    json.dump(results, fh, indent=2)
print("Wrote work/outputs/validation_metrics.json")

Wrote work/outputs/validation_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.